# Goal 1：真实角色记录与离线复算

教师映射：`任务3_LLM辅助评估清洗.ipynb` → 本入口。默认不会调用模型。LIVE、REPLAY、MOCK_TEST 明确分开；工程测试不冒充自然Agent失败或真实Human–AI争论。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from task1.workflow.io import DATA, CONFIG, EVIDENCE, read_json, digest
from task1.workflow.diagnostics import profile, aggregate, time_boundaries, independent_profile_review
from task1.workflow.tools import execute_tool
policy=read_json(CONFIG); pilot=read_json(ROOT/'task1/config/pilot.json')
assert digest(DATA)==policy['raw_sha256']==pilot['raw_sha256']
raw=read_json(DATA)
print('模式：RECOMPUTE；本次模型调用：0')
print('原始输入 SHA256:', digest(DATA))
print('固定开发样本:', pilot['ids'])

模式：RECOMPUTE；本次模型调用：0
原始输入 SHA256: c59be4c079d2ffd8ae0127c8a361c77ba277252abb2c7202efb9ee4e8e6084d3
固定开发样本: ['0', '1', '2', '246', '256', '306', '352']


In [2]:
MODE='RECOMPUTE'
RUN_ID='g1-live-20260926-03'
ENABLE_LIVE=False
assert MODE in ('RECOMPUTE','LIVE')

## 真实运行与失败边界
run03仅 A 研究角色成功并执行 source_evidence；B 因 workspace routing discovery failed 失败，C 未调用，反馈循环未完成。run01/02失败均保留。内部重连预算问题使所有新LIVE被冻结。不能把下方离线复算当作补齐三个真实角色。

In [3]:
directory=EVIDENCE/'runs'/RUN_ID
manifest=read_json(directory/'manifest.json')
print('run',RUN_ID,'code',manifest['code_sha'],'mode',manifest['mode'],'status',manifest['status'])
for call in manifest['calls']:
    reply=read_json(directory/call['response'])
    print(call['role'],call['call_id'],call['thread_id'],reply['action'],reply['feedback_refs'])
for tool in manifest['tools']: print('tool',tool['tool_id'],tool['action'],tool['status'])

run g1-live-20260926-03 code d1444eb07035ef651a36a3e89269f004186581df mode LIVE status BLOCKED
research g1-live-20260926-03-01-research 01a0dd66-2d69-7123-9ea2-b8ec2b958dd3 source_evidence []
tool 0ad49e824507fb5f5ee9e1d042b423f1cee074df1d78283b5385d277638fa15c source_evidence EXECUTED


## 按已记录代码版本复算实际动作
控制器已在失败后修复，当前源hash与旧run不同。下面从已存在的Git CODE_SHA恢复任务源码到临时目录，校验hash，再真实重跑旧run唯一成功动作 source_evidence。不会切换当前分支，也不修改旧产物。它只能证明这一项复算一致。

In [4]:
if MODE=='RECOMPUTE':
    from task1.scripts.recompute_archived_run import recompute_archived
    result=recompute_archived(RUN_ID)
    assert result['new_model_calls']==0 and result['status']=='VERIFIED'
    assert [c['action'] for c in result['checks']]==['source_evidence']
    print(json.dumps(result,ensure_ascii=False,indent=2))
else:
    assert ENABLE_LIVE, '必须显式启用LIVE'
    assert policy['live_execution_enabled'], policy['live_stop_reason']
    assert not directory.exists(), '新模型实验须使用新的run_id；不得重置总预算'
    from task1.workflow.controller import Controller
    result=Controller(RUN_ID).run()
    print(result['status'])

{
  "mode": "REPLAY_RECOMPUTE",
  "new_model_calls": 0,
  "source_live_run": "g1-live-20260926-03",
  "input_sha256": "c59be4c079d2ffd8ae0127c8a361c77ba277252abb2c7202efb9ee4e8e6084d3",
  "source_code_sha": "d1444eb07035ef651a36a3e89269f004186581df",
  "recomputed_at": "2026-09-26T11:27:51.500683+00:00",
  "checks": [
    {
      "tool_id": "0ad49e824507fb5f5ee9e1d042b423f1cee074df1d78283b5385d277638fa15c",
      "action": "source_evidence",
      "expected_sha256": "107616dce9e51de0b85e18ac5364a04c738fd2ef4e04028215c4ecb0ee4cc182",
      "recomputed_sha256": "107616dce9e51de0b85e18ac5364a04c738fd2ef4e04028215c4ecb0ee4cc182",
      "equal": true
    }
  ],
  "status": "VERIFIED",
  "meaning": "Tools genuinely reran on the raw JSON; not a new model experiment or quality approval",
  "restoration": "Tracked workflow/config from existing code_sha; current workspace not checked out or changed",
  "scope": "Only saved successful tool actions. This partial run has source_evidence only; no co

## 当前工程工具的真实数值复算（不是 Agent 后续动作）
直接从固定7例原始输入重算 profile 与独立核验，再对照保存结果。该计算由Notebook发起，不伪称B/C模型完成了任务。

In [5]:
numeric=execute_tool('profile_pilot',pilot['ids'],policy)
rows=numeric['result']['profiles']
review=execute_tool('verify_profiles',pilot['ids'],policy,rows)
recheck=execute_tool('recompute_check',pilot['ids'],policy,rows)
assert review['status']==recheck['status']=='VERIFIED'
assert rows==read_json(ROOT/'task1/results/goal1/pilot_diagnostics.json')['profiles']
print(numeric['input_records'],numeric['input_points'],review['status'],recheck['result']['exact_match'])

7 783 VERIFIED True


## 状态解释
VERIFIED 仅为结构/复算有效；没有冻结质量阈值，也没有合法的真实三步全链结果，因此 QUALITY_ACCEPTED 不成立。最终 GPT second review=PENDING，Submission=NOT_READY。

In [6]:
checkpoint=read_json(directory/'checkpoint.json')
assert checkpoint['accepted_version'] is None
print('feedback decisions:',json.dumps(checkpoint['decisions'],ensure_ascii=False,indent=2))
print('raw hash unchanged:',digest(DATA)==policy['raw_sha256'])

feedback decisions: [
  {
    "decision_id": "g1-live-20260926-03-01-research-decision",
    "trigger_call": "g1-live-20260926-03-01-research",
    "trigger_tool": "0ad49e824507fb5f5ee9e1d042b423f1cee074df1d78283b5385d277638fa15c",
    "status": "NEEDS_REVIEW",
    "reason": "Engineering diagnostics executed; real baseline semantic blockers and quality thresholds unresolved",
    "quality_acceptance": "PENDING_RESEARCH_REVIEW",
    "next_task": "请求核查绑定此数据文件的 CRS、距离策略及其批准依据，并核查方向规则的执行顺序、重复轮次、末端和零位移处理。执行角色可对全部固定 pilot 开展 profile_pilot、time_boundaries、duplicate_details，核验不依赖距离的原始诊断；保留输入顺序、原值和原始索引，分别检查重复时间、同刻异位、连续同位及长时间间隔，不自动删除或跨记录拼接。"
  }
]
raw hash unchanged: True
